# 第 18 章 OpenCV 攝影機即時影像分析

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from opencv_tools import opencv_tools # 匯入封裝的功能
import time

## 18-1-4 程式範例：取得即時畫面並顯示

In [2]:
cap = cv2.VideoCapture(0) # 0 通常代表預設攝影機
# cap = cv2.VideoCapture('sample/video/me.mp4') # 測試用影片

while True:
    ret, frame = cap.read()
    if not ret:
        print('無法讀取攝影機畫面')
        break

    cv2.imshow('Webcam', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()  # 關閉攝影機
cv2.destroyAllWindows()

無法讀取攝影機畫面


### 18-1-5 進階程式範例：處理串流不穩的情況

In [3]:
cap = cv2.VideoCapture(0)
last_frame_time = time.time()  # 記錄最後一次成功讀取的時間
TIMEOUT_LIMIT = 5  # 設定斷線容忍時間（秒）
while True:
    ret, frame = cap.read()
    if ret:
        # 如果讀取成功，更新時間戳記
        last_frame_time = time.time()
        cv2.imshow('Webcam', frame)
    else:
        # 如果讀取失敗，檢查是否超過容忍時間
        elapsed_time = time.time() - last_frame_time
        if elapsed_time > TIMEOUT_LIMIT:
            print("\n[ERROR] 串流中斷過久，終止程式。")
            break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### 18-2-3 程式範例：即時邊緣偵測

In [4]:
def process_image(frame):
    start_time = time.time()  # 記錄開始時間
    # 調整亮度與對比度（改善偏暗、層次不夠分明的問題）
    brightened = cv2.convertScaleAbs(frame, alpha=1.0, beta=60)
    # 水平翻轉（做出符合直覺的視角）
    flipped = cv2.flip(brightened, 1)
    
    processed_time = time.time() - start_time  # 計算處理時間
    print(f"Process time = {processed_time}")
    return flipped

In [5]:
# 若是外接鏡頭，參數可能為其他整數 
cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture('sample/video/me.mp4') # 測試用影片

while True:
    # 逐幀讀取即時畫面
    ret, frame = cap.read()
    if not ret:
        print('無法讀取攝影機畫面')
        break
    
    # 呼叫自訂處理函式
    result_frame = process_image(frame)
    
    # 顯示處理後的結果
    cv2.imshow('Live Video (Bright + Mirror)', result_frame)
    
    # 按下 'q' 鍵退出程式
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放攝影機並關閉視窗
cap.release()
cv2.destroyAllWindows()

Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0005099773406982422
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0010008811950683594
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0


### 18-2-5 進階程式範例：跳幀處理 (降低 fps) 

In [6]:
# 若是外接鏡頭，參數可能為其他整數 
cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture('sample/video/me.mp4') # 測試用影片

# 定義初始變數
frame_count = 0       # 用來記錄目前跑到第幾張影格
process_interval = 6  # 設定每 6 張影格處理 1 張（相當於 30 fps 降至 5 fps）

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame_count += 1  # 每次讀取到新影格，計數器就加 1
    
    if frame_count % process_interval == 0:
        # 每 6 張才處理 1 張，並更新顯示
        result_frame = process_image(frame)
        cv2.imshow('Live Video (Bright + Mirror)', result_frame)
    # 跳過的影格不做任何事，畫面會停留在上一次處理的結果
    
    # 按下 'q' 鍵退出程式
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放攝影機並關閉視窗
cap.release()
cv2.destroyAllWindows()

Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0
Process time = 0.0


### 18-E-1 程式範例：製作即時素描風格效果

In [7]:
def process_image(frame):
    # 沿用前面的處理：先調亮並翻轉
    brightened = cv2.convertScaleAbs(frame, alpha=1.2, beta=50)
    flipped = cv2.flip(brightened, 1)
    
    # 轉灰階後進行 Sobel 邊緣偵測
    gray = cv2.cvtColor(flipped, cv2.COLOR_BGR2GRAY)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel = cv2.convertScaleAbs(sobel_x) + cv2.convertScaleAbs(sobel_y)
    
    # 反轉色彩，做出白底黑線的素描風格
    sketch = cv2.bitwise_not(sobel)
    return sketch

In [8]:
# 若是外接鏡頭，參數可能為其他整數 
cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture('sample/video/me.mp4') # 測試用影片

while True:
    # 逐幀讀取即時畫面
    ret, frame = cap.read()
    if not ret:
        print('無法讀取攝影機畫面')
        break
    
    # 呼叫自訂處理函式
    result_frame = process_image(frame)
    
    # 顯示處理後的結果
    cv2.imshow('Live Video (Bright + Mirror)', result_frame)
    
    # 按下 'q' 鍵退出程式
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 釋放攝影機並關閉視窗
cap.release()
cv2.destroyAllWindows()